# Diagnóstico de Cáncer de Mama (Wisconsin)

Proyecto de clasificación listo para **Google Colab**.

**Pregunta:** ¿podemos predecir si un tumor es maligno o benigno a partir de características de células?

**Target:** `diagnosis` (M = maligno, B = benigno)

**Stack:** pandas, seaborn, scikit-learn, shap

Dataset clásico Breast Cancer Wisconsin (UCI / sklearn).

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, RocCurveDisplay

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
print("OK")

## 1. Cargar datos

In [ ]:
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df["diagnosis"] = data.target  # 0 = maligno, 1 = benigno
print(df.shape)
print(df["diagnosis"].value_counts())
df.head()

## 2. EDA

In [ ]:
print("Proporción benigno (1):", df["diagnosis"].mean().round(3))
df.describe().T.head(10)

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="diagnosis")
plt.xticks([0, 1], ["Maligno", "Benigno"])
plt.title("Distribución de diagnóstico")
plt.show()

In [ ]:
features_plot = ["mean radius", "mean texture", "mean perimeter", "mean area", "mean smoothness"]
fig, axes = plt.subplots(1, 5, figsize=(16, 3))
for ax, col in zip(axes, features_plot):
    sns.boxplot(data=df, x="diagnosis", y=col, ax=ax)
    ax.set_xticklabels(["M", "B"])
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
corr = df.corr()
sns.heatmap(corr[["diagnosis"]].sort_values("diagnosis", ascending=False).head(15), annot=True, cmap="coolwarm")
plt.title("Correlación con diagnosis")
plt.show()

### Hallazgos
- Dataset balanceado (~63% benigno).
- Features de tamaño (radius, perimeter, area) correlacionan fuerte con diagnóstico.
- Tumores malignos tienden a ser más grandes.

## 3. Preparar datos

In [ ]:
X = df.drop(columns="diagnosis")
y = df["diagnosis"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print(X_train.shape, X_test.shape)

## 4. Modelos

In [ ]:
modelos = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "SVM": SVC(probability=True, random_state=RANDOM_STATE),
}

resultados = []
pipes = {}

for nombre, clf in modelos.items():
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", clf)
    ])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    resultados.append({
        "modelo": nombre,
        "accuracy": accuracy_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
    })
    pipes[nombre] = pipe

pd.DataFrame(resultados).sort_values("roc_auc", ascending=False)

## 5. Evaluación

In [ ]:
mejor = "Random Forest"
pipe = pipes[mejor]
y_pred = pipe.predict(X_test)
y_proba = pipe.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=["Maligno", "Benigno"]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["Maligno", "Benigno"]).plot(ax=axes[0], cmap="Blues")
axes[0].set_title("Matriz de confusión")

RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1])
axes[1].set_title("Curva ROC")
plt.tight_layout()
plt.show()

## 6. Feature importance

In [ ]:
rf = pipes["Random Forest"].named_steps["model"]
importancias = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(15)

plt.figure(figsize=(8, 6))
sns.barplot(x=importancias.values, y=importancias.index)
plt.title("Top 15 features — Random Forest")
plt.tight_layout()
plt.show()

## 7. SHAP

In [ ]:
!pip install -q shap

In [ ]:
import shap

rf_pipe = pipes["Random Forest"]
X_train_scaled = rf_pipe.named_steps["scaler"].transform(X_train)
model = rf_pipe.named_steps["model"]

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_train_scaled)

print("SHAP listo")

In [ ]:
shap.summary_plot(shap_values[1], X_train_scaled, feature_names=X.columns, max_display=12, show=False)
plt.title("SHAP Summary — clase benigno")
plt.tight_layout()
plt.show()

In [ ]:
shap.summary_plot(shap_values[1], X_train_scaled, feature_names=X.columns, plot_type="bar", max_display=12, show=False)
plt.title("Importancia media SHAP")
plt.tight_layout()
plt.show()

## 8. Conclusiones

- Accuracy y ROC-AUC suelen superar **0.95** en test.
- Features de tamaño y textura son las más importantes.
- Dataset limpio y bien conocido: excelente para practicar clasificación binaria y explicabilidad.

**Rango esperado:** Accuracy ~0.95–0.98 | ROC-AUC ~0.97–0.99